# STAGE A2 SEED 42 — CANONICAL FRESH GOOGLE COLAB TRAINING NOTEBOOK
**Protocol**: Stage A2 Protocol V1.5 (Amendment 12 Locked)  
**Execution Scope**: 35,000 Train Sessions (586,577 events) | 7,500 Val Sessions (119,531 events)  
**Durable Storage Root**: `/content/drive/MyDrive/Chuyende-stage-a2/runs/HDFS`  
**Target Seed**: 42 (Fresh start from Epoch 1, Step 0 with exact canonical RNG ordering)  

### Instructions:
1. Connect to any GPU Runtime (`Runtime` -> `Change runtime type` -> `T4` / `L4` / `A100`).
2. Click **Run all** (`Runtime` -> `Run all` or `Ctrl+F9`).
3. Click **Connect to Google Drive** when prompted in Cell 1.
4. Cell 3 performs automated bootstrap, deterministic qualification, authorization, and dry-run.
5. Cell 4 archives any existing noncanonical run and streams fresh Seed 42 training live to completion.


In [ ]:
# CELL 1 — CONFIG + GOOGLE DRIVE + GPU DISCOVERY
import os, sys, subprocess
from pathlib import Path
from google.colab import drive

# 1. Canonical Configuration Constants
EXECUTION_COMMIT = "c6d9805ae4dd9d3f6740222ec1eb3ec98554aeb6"
REPO = "/content/Research"
DRIVE_ROOT = "/content/drive/MyDrive/Chuyende-stage-a2"
DURABLE_RUN_ROOT = "/content/drive/MyDrive/Chuyende-stage-a2/runs/HDFS"
DATASET_DRIVE = "/content/drive/MyDrive/Chuyende-stage-a2/datasets/HDFS_1.tar.gz"
DATASET_LOCAL = "/content/stage-a2-data/HDFS_1.tar.gz"
EXPECTED_HDFS_SHA = "6ca6c5bc2671c66afecee9369a2fdac606bf33997a2494ac66aa411fe3e95169"
SEED = 42

# 2. Environment Variables for Determinism & Unbuffered Logging
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONUNBUFFERED"] = "1"

# 3. Mount Google Drive
drive_mount_point = Path("/content/drive")
drive.mount(str(drive_mount_point), force_remount=False)

drive_root_p = Path(DRIVE_ROOT)
drive_root_p.mkdir(parents=True, exist_ok=True)
assert drive_root_p.exists(), f"FATAL: Durable Drive root missing at {drive_root_p}"
print("Google Drive Mounted Successfully:", drive_root_p)

# 4. Fail-Closed GPU Discovery via nvidia-smi
try:
    smi_out = subprocess.check_output(["nvidia-smi"], text=True)
    print("NVIDIA Driver & Hardware Detected:")
    print(smi_out.strip())
except Exception as e:
    raise RuntimeError(
        "FATAL: No NVIDIA GPU detected via nvidia-smi! "
        "Please select a GPU runtime in Colab: Runtime -> Change runtime type -> T4 / L4 / A100."
    ) from e

gpu_info = subprocess.check_output([
    "nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"
], text=True).strip()
print("\nAssigned GPU Hardware:", gpu_info)
print("Discovery Complete. Ready for Clean Environment Setup.")


In [ ]:
# CELL 2 — CLEAN EXECUTION ENVIRONMENT + DATASET INTEGRITY
import os, sys, subprocess, shutil, hashlib, re
from pathlib import Path

def compute_sha256_streaming(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    hasher = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(chunk_size):
            hasher.update(chunk)
    return hasher.hexdigest()

repo_dir = Path(REPO)
repo_url = "https://github.com/Minhlike/Chuyende.git"

# 1. Clean Fresh Clone of Repository
if repo_dir.exists():
    print(f"Removing existing {repo_dir} for clean fresh clone...")
    shutil.rmtree(repo_dir)

print(f"Cloning clean repository from {repo_url}...")
subprocess.run(["git", "clone", repo_url, str(repo_dir)], check=True)

# 2. Detached Checkout of Approved Fix Commit
print(f"Detached checkout of approved execution commit: {EXECUTION_COMMIT}")
subprocess.run(["git", "checkout", EXECUTION_COMMIT.strip()], cwd=str(repo_dir), check=True)

head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=str(repo_dir), text=True).strip()
assert head == EXECUTION_COMMIT.strip(), f"Commit mismatch: {head} != {EXECUTION_COMMIT}"
print("Approved Execution Source Verified at HEAD:", head)

# 3. Install Repository Dependencies
print("Installing editable repository dependencies via pyproject.toml...")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], cwd=str(repo_dir), check=True)

# 4. Verify PyTorch 2.6.0+cu124 and CUDA 12.4
def verify_torch_runtime():
    cmd = [
        sys.executable, "-c",
        "import torch; "
        "assert torch.__version__ == '2.6.0+cu124', f'Torch version: {torch.__version__} != 2.6.0+cu124'; "
        "assert torch.version.cuda == '12.4', f'CUDA runtime: {torch.version.cuda} != 12.4'; "
        "assert torch.cuda.is_available() is True, 'CUDA unavailable!'"
    ]
    return subprocess.run(cmd, capture_output=True, text=True)

res = verify_torch_runtime()
if res.returncode != 0:
    print("Reinstalling official PyTorch 2.6.0+cu124 wheel...")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--no-cache-dir", "--force-reinstall",
        "torch==2.6.0", "--index-url", "https://download.pytorch.org/whl/cu124"
    ], check=True)
    res_re = verify_torch_runtime()
    if res_re.returncode != 0:
        raise RuntimeError(f"FATAL: Torch verification failed: {res_re.stderr}")

print("Exact PyTorch Runtime Verified: torch 2.6.0+cu124 (CUDA 12.4, cuda.is_available=True)")

# 5. Fail-Closed Dataset Streaming Copy & SHA-256 Verification
drive_canonical = Path(DATASET_DRIVE)
drive_fallback = Path("/content/drive/MyDrive/HDFS_1.tar.gz")

if drive_canonical.exists():
    drive_src = drive_canonical
elif drive_fallback.exists():
    drive_src = drive_fallback
else:
    raise FileNotFoundError(f"FATAL: HDFS dataset missing on Drive! Checked: {drive_canonical}, {drive_fallback}")

print(f"Drive Dataset Source: {drive_src}")
src_sha = compute_sha256_streaming(drive_src)
print(f"Drive Source SHA-256: {src_sha}")
assert src_sha == EXPECTED_HDFS_SHA, f"Drive source SHA mismatch: {src_sha} != {EXPECTED_HDFS_SHA}"

local_dest = Path(DATASET_LOCAL)
local_dest.parent.mkdir(parents=True, exist_ok=True)
tmp_dest = Path(DATASET_LOCAL + ".tmp")

print(f"Copying Drive source to local tmp file: {tmp_dest}...")
shutil.copy2(drive_src, tmp_dest)
tmp_sha = compute_sha256_streaming(tmp_dest)
assert tmp_sha == EXPECTED_HDFS_SHA, f"Temporary file SHA mismatch: {tmp_sha} != {EXPECTED_HDFS_SHA}"

os.replace(tmp_dest, local_dest)
final_local_sha = compute_sha256_streaming(local_dest)
print(f"Canonical Local SHA-256: {final_local_sha}")
assert final_local_sha == EXPECTED_HDFS_SHA, f"Final local SHA mismatch: {final_local_sha} != {EXPECTED_HDFS_SHA}"
assert local_dest.stat().st_size == drive_src.stat().st_size, "File size mismatch!"
print("HDFS Raw Dataset Parity Verified: 100% MATCH (6ca6c5bc2671c66a...)")


In [ ]:
# CELL 3 — BOOTSTRAP + DETERMINISTIC QUALIFICATION + AUTHORIZATION + DRY-RUN
import os, sys, subprocess, json, hashlib
from datetime import datetime, timezone
from pathlib import Path

def compute_sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

repo_dir = Path(REPO)
local_data = Path(DATASET_LOCAL)
durable_root = Path(DRIVE_ROOT)
durable_runs = Path(DURABLE_RUN_ROOT)
env_lock_output = repo_dir / "experiments" / "evidence" / "stage-a2" / "preexecution" / "STAGE-A2-COLAB-EXECUTION-ENVIRONMENT-V1.5.json"
output_dir = repo_dir / "experiments" / "evidence" / "stage-a2" / "implementation"
plan_path = repo_dir / "experiments" / "plans" / "STAGE-A2-FIVE-SEED-EXECUTION-PLAN-V1.5.json"
auth_dest = repo_dir / "experiments" / "evidence" / "stage-a2" / "preexecution" / f"SEED{SEED}-COLAB-LAUNCH-AUTHORIZATION-V1.5.json"

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONUNBUFFERED"] = "1"

# A. Bootstrap: Dynamic hardware environment lock generation
print("--- [A] RUNNING BOOTSTRAP ---")
bootstrap_cmd = [
    sys.executable, "scripts/bootstrap_stage_a2_colab.py",
    "--repo-dir", str(repo_dir),
    "--drive-data-source", str(DATASET_DRIVE),
    "--local-data-dest", str(local_data),
    "--durable-root", str(durable_root),
    "--env-lock-output", str(env_lock_output)
]
subprocess.run(bootstrap_cmd, cwd=str(repo_dir), check=True)
assert env_lock_output.exists(), f"Environment lock was not generated at {env_lock_output}"
print("Bootstrap Complete. Generated Environment Lock Candidate:", env_lock_output)

# B. Deterministic Qualification (0 Real Optimizer Steps)
print("\n--- [B] RUNNING DETERMINISTIC QUALIFICATION ---")
qual_cmd = [
    sys.executable, "scripts/run_stage_a2_deterministic_qualification.py",
    "--device", "cuda",
    "--base-dir", str(repo_dir),
    "--environment-lock", str(env_lock_output),
    "--output-dir", str(output_dir)
]
subprocess.run(qual_cmd, cwd=str(repo_dir), check=True)

resume_p = output_dir / "DETERMINISTIC-RESUME-EVIDENCE.json"
resume_data = json.loads(resume_p.read_text(encoding="utf-8"))
assert resume_data["qualification_status"] == "PASS", f"Qualification status: {resume_data['qualification_status']}"
assert resume_data["fresh_process_isolated"] is True, "Resume not executed in isolated fresh process!"
assert resume_data["execution_code_commit_sha"] == EXECUTION_COMMIT, "Qualification commit mismatch!"
print("Deterministic Qualification: PASS (0 Real Optimizer Steps, Numerical Divergence = 0.0)")

# C. Generate Launch Authorization matching verify_preflight schema
print("\n--- [C] GENERATING LAUNCH AUTHORIZATION ---")
auth_data = {
    "authorization_id": f"AUTH-STAGE-A2-HDFS-SEED{SEED}-COLAB-V1.5",
    "authorized_at": datetime.now(timezone.utc).isoformat(),
    "stage": "STAGE_A2",
    "dataset": "HDFS",
    "split_id": "SPL-HDFS-001",
    "seed": SEED,
    "authorization_status": "AUTHORIZED",
    "execution_provider": "GOOGLE_COLAB",
    "expected_execution_code_commit_sha": EXECUTION_COMMIT,
    "execution_plan_path": "experiments/plans/STAGE-A2-FIVE-SEED-EXECUTION-PLAN-V1.5.json",
    "execution_plan_sha256": compute_sha256(plan_path),
    "environment_lock_path": "experiments/evidence/stage-a2/preexecution/STAGE-A2-COLAB-EXECUTION-ENVIRONMENT-V1.5.json",
    "environment_lock_sha256": compute_sha256(env_lock_output),
    "raw_hdfs_sha256": EXPECTED_HDFS_SHA,
    "train_membership_sha256": "65b76694b0a3cf5c6d684a26899b1e5dca634cfd0985560149feddc12ca8ccfc",
    "val_membership_sha256": "14cf689f9682a354e104463b9f02806629a683dfdf36d72d88daf5b407b0609a",
    "train_sessions_count": 35000,
    "val_sessions_count": 7500,
    "train_events_count": 586577,
    "val_events_count": 119531,
    "train_graph_events_count": 586577,
    "val_graph_events_count": 119531,
    "train_windows_count": 2292,
    "val_windows_count": 467,
    "optimizer_steps_per_epoch": 573,
    "test_opened": False,
    "firewall_policy": "TEST_SET_SEALED_UNTIL_STAGE_B",
    "authorized_by": "QUALIFICATION_EVIDENCE_AUDIT_PASS"
}
auth_dest.parent.mkdir(parents=True, exist_ok=True)
auth_dest.write_text(json.dumps(auth_data, indent=2) + "\n", encoding="utf-8")
print("Launch Authorization Created:", auth_dest)

# D. Canonical Dry-Run Verification (0 Optimizer Steps)
print("\n--- [D] EXECUTING CANONICAL PREFLIGHT DRY-RUN ---")
dry_cmd = [
    sys.executable, "-u", "scripts/run_stage_a2_five_seed_empirical.py",
    "--seed", str(SEED),
    "--dry-run",
    "--base-dir", str(repo_dir),
    "--dataset-path", str(local_data),
    "--durable-root", str(durable_runs),
    "--plan", str(plan_path),
    "--environment-lock", str(env_lock_output),
    "--authorization", str(auth_dest)
]
proc = subprocess.run(dry_cmd, cwd=str(repo_dir), capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print("Dry-run STDERR:\n", proc.stderr)
    raise RuntimeError(f"FATAL: Seed {SEED} dry-run failed with code {proc.returncode}")

assert "OptimizerStepsExecuted=0" in proc.stdout or "Optimizer Steps Executed: 0" in proc.stdout, "Dry run optimizer steps != 0!"
assert "TEST_OPENED=false" in proc.stdout or "Connected Test Firewall: LOCKED" in proc.stdout, "Test firewall breached!"

PREFLIGHT_PASSED = True

print("========================================")
print("STAGE A2 PRE-FLIGHT: PASS")
print("QUALIFICATION: PASS")
print("DATASET: PASS")
print("RNG FIX COMMIT: PASS")
print("OPTIMIZER STEPS EXECUTED: 0")
print("READY FOR FRESH CANONICAL SEED 42")
print("========================================")


In [ ]:
# CELL 4 — PRESERVE OLD RUN + LAUNCH REAL CANONICAL SEED 42 TRAINING
import os, sys, subprocess, json, shutil, hashlib
from datetime import datetime, timezone
from pathlib import Path

def compute_sha256_streaming(path: Path) -> str:
    hasher = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(8 * 1024 * 1024):
            hasher.update(chunk)
    return hasher.hexdigest()

# 1. Verify Preflight Gate State
if "PREFLIGHT_PASSED" not in globals() or not PREFLIGHT_PASSED:
    raise RuntimeError("FATAL: Cell 3 preflight verification did not pass in this session!")

repo_dir = Path(REPO)
local_data = Path(DATASET_LOCAL)
durable_runs = Path(DURABLE_RUN_ROOT)
durable_seed_dir = durable_runs / f"seed-{SEED}"
plan_path = repo_dir / "experiments" / "plans" / "STAGE-A2-FIVE-SEED-EXECUTION-PLAN-V1.5.json"
env_lock_output = repo_dir / "experiments" / "evidence" / "stage-a2" / "preexecution" / "STAGE-A2-COLAB-EXECUTION-ENVIRONMENT-V1.5.json"
auth_dest = repo_dir / "experiments" / "evidence" / "stage-a2" / "preexecution" / f"SEED{SEED}-COLAB-LAUNCH-AUTHORIZATION-V1.5.json"

EXPECTED_OLD_CHECKPOINT_SHA = "cac60f9f64c1e0ccbf87cc326aef384df31d0784a1225bea89ac0d108f29d372"

# 2. Archive and Move Old Noncanonical Seed 42 Run Directory (No Deletion)
if durable_seed_dir.exists():
    utc_ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    forensic_dest = durable_runs / f"seed-42-forensic-noncanonical-{utc_ts}"
    print(f"Archiving pre-existing Seed 42 directory:\n  FROM: {durable_seed_dir}\n  TO:   {forensic_dest}")
    
    durable_seed_dir.rename(forensic_dest)
    assert forensic_dest.exists(), f"Failed to rename old directory to {forensic_dest}"
    assert not durable_seed_dir.exists(), f"Old directory still exists at {durable_seed_dir}"
    
    # Check old checkpoint SHA if present
    old_ckpt = forensic_dest / "last_checkpoint.pt"
    if old_ckpt.exists():
        actual_old_sha = compute_sha256_streaming(old_ckpt)
        print(f"Old Checkpoint Preserved (SHA-256: {actual_old_sha})")
        if actual_old_sha == EXPECTED_OLD_CHECKPOINT_SHA:
            print("Old Checkpoint SHA-256 matches expected historical artifact.")
            
    # Write RUN-CLASSIFICATION.json into forensic directory
    class_file = forensic_dest / "RUN-CLASSIFICATION.json"
    class_data = {
        "seed": 42,
        "classification": "NONCANONICAL_RNG_INITIALIZATION",
        "reason": "model initialized before canonical seed establishment",
        "archived_at": datetime.now(timezone.utc).isoformat(),
        "preserve": {
            "old_completed_epoch": 1,
            "old_global_step": 573
        }
    }
    class_file.write_text(json.dumps(class_data, indent=2) + "\n", encoding="utf-8")
    print(f"Forensic Classification Written: {class_file}")

# 3. Ensure Local Workspace Cleanliness for Fresh Launch
local_run_dir = repo_dir / "experiments" / "runs" / "stage-a2" / "HDFS" / f"seed-{SEED}"
local_art_dir = repo_dir / ".artifacts" / "stage-a2" / "HDFS" / f"seed-{SEED}"

if local_run_dir.exists():
    shutil.rmtree(local_run_dir)
if local_art_dir.exists():
    shutil.rmtree(local_art_dir)

# 4. Construct Real Canonical Training Command
real_train_cmd = [
    sys.executable, "-u", "scripts/run_stage_a2_five_seed_empirical.py",
    "--seed", str(SEED),
    "--authorize-real-empirical-execution",
    "--base-dir", str(repo_dir),
    "--dataset-path", str(local_data),
    "--durable-root", str(durable_runs),
    "--plan", str(plan_path),
    "--environment-lock", str(env_lock_output),
    "--authorization", str(auth_dest)
]

print("=================================================================")
print("   LAUNCHING REAL FRESH CANONICAL SEED 42 TRAINING               ")
print("=================================================================")
print("Command:", " ".join(real_train_cmd))
print("Live Output Streaming Starting Below:\n")

# 5. Launch and Stream Live Output Unbuffered
proc = subprocess.Popen(
    real_train_cmd,
    cwd=str(repo_dir),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in proc.stdout:
    print(line, end="", flush=True)

proc.wait()
assert proc.returncode == 0, f"FATAL: Seed {SEED} training failed with return code {proc.returncode}"
print(f"\n[COMPLETE] Seed {SEED} Training Exited Successfully with Code 0.")
